# 🏆 Churn Prediction: Telco Production Pipeline (V7.0 - COST SENSITIVE / F2-OPTIMIZED)
**Dataset**: [blastchar/telco-customer-churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)
**Focus**: SMOTE + F2-Score Optimization (Prioritizing Recall 2:1 over Precision).

In [ ]:
# 1. Setup & Auth
!pip uninstall umap-learn hdbscan -y -q
!pip install opendatasets mlflow xgboost shap python-dotenv seaborn dagshub scikit-learn==1.5.1 imbalanced-learn -q

import sklearn; print(f"✅ Environment Ready: {sklearn.__version__}")
import os, pandas as pd, numpy as np, mlflow, mlflow.sklearn, matplotlib.pyplot as plt, seaborn as sns, shap, dagshub
import opendatasets as od
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, precision_recall_curve, recall_score, precision_score, make_scorer, fbeta_score
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

mlflow.sklearn.autolog(log_models=False)
dagshub.init(repo_owner="nhannhb92", repo_name="msa24-ddm501-group6-final-project", mlflow=True)

In [ ]:
# 2. Telco Specific Cleaning
def clean_telco_data(df):
    df.columns = [col.lower() for col in df.columns]
    df = df.drop(columns=[c for c in ['customerid'] if c in df.columns])
    df['totalcharges'] = pd.to_numeric(df['totalcharges'], errors='coerce')
    if 'churn' in df.columns:
        df['churn'] = df['churn'].map({'Yes': 1, 'No': 0})
    return df.dropna()

if not os.path.exists("telco-customer-churn"):
    od.download("https://www.kaggle.com/datasets/blastchar/telco-customer-churn")

df_raw = clean_telco_data(pd.read_csv("telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"))
df_train, df_test = train_test_split(df_raw, test_size=0.2, random_state=42, stratify=df_raw['churn'])


In [ ]:
# 4. Custom Scoring Logic: F2-Score (Weights Recall twice as high as Precision)
def f2_threshold_scorer(y_true, y_probs):
    """Finds the max F2 score available across all thresholds for this specific model configuration."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    # F2 = 5 * (P * R) / (4*P + R)
    f2_scores = 5 * (precisions * recalls) / (4 * precisions + recalls + 1e-10)
    return f2_scores.max()

f2_scorer = make_scorer(f2_threshold_scorer, needs_proba=True)

# 5. Training Engine
def run_telco_experiment(df_tr, df_te, model_type):
    X_tr, y_tr = df_tr.drop(columns=['churn']), df_tr['churn']
    X_te, y_te = df_te.drop(columns=['churn']), df_te['churn']
    
    num_f = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    cat_f = X_tr.select_dtypes(include=['object']).columns.tolist()
    
    pre = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_f),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_f)
    ])
    
    if model_type == "xgboost":
        clf = XGBClassifier(n_estimators=100, max_depth=5, eval_metric='logloss')
        m_p = {'clf__max_depth': [3, 5, 7]}
    elif model_type == "random_forest":
        clf = RandomForestClassifier(random_state=42)
        m_p = {'clf__n_estimators': [100, 200]}
    else:
        clf = LogisticRegression(max_iter=2000)
        m_p = {'clf__C': [0.1, 1.0, 10.0]}
        
    mlflow.set_experiment("Churn_F2_Optimization")
    with mlflow.start_run(run_name=f"F2_OPT_{model_type.upper()}"):
        pipe = ImbPipeline([
            ('pre', pre), 
            ('smote', SMOTE(random_state=42)), 
            ('clf', clf)
        ])
        
        # OPTIMIZE FOR F2-SCORE
        search = RandomizedSearchCV(pipe, m_p, n_iter=3, cv=3, scoring=f2_scorer, verbose=1)
        search.fit(X_tr, y_tr)
        model = search.best_estimator_
        
        # --- FIND OPTIMAL THRESHOLD FOR F2 ---
        y_prb = model.predict_proba(X_te)[:, 1]
        precisions, recalls, thresholds = precision_recall_curve(y_te, y_prb)
        f2_scores = 5 * (precisions * recalls) / (4 * precisions + recalls + 1e-10)
        best_idx = np.argmax(f2_scores)
        best_threshold = thresholds[min(best_idx, len(thresholds)-1)]
        
        y_prd_custom = (y_prb >= best_threshold).astype(int)
        
        print(f"\n🏆 F2-OPTIMIZED REPORT ({model_type.upper()}):")
        print(f"Best F2 during search: {search.best_score_:.4f}")
        print(f"Optimal F2 Threshold: {best_threshold:.4f}")
        print(classification_report(y_te, y_prd_custom))
        
        mlflow.log_metric("f2_score", fbeta_score(y_te, y_prd_custom, beta=2))
        mlflow.log_metric("recall", recall_score(y_te, y_prd_custom))
        mlflow.log_metric("precision", precision_score(y_te, y_prd_custom))
        mlflow.log_param("threshold", best_threshold)
        
        # RAI Artifacts
        try:
            X_t = model.named_steps['pre'].transform(X_te)
            if hasattr(X_t, "toarray"): X_t = X_t.toarray()
            explainer = shap.Explainer(model.named_steps['clf'])
            shap_v = explainer(X_t)
            plt.figure(figsize=(10, 6))
            shap.summary_plot(shap_v, X_t, feature_names=model.named_steps['pre'].get_feature_names_out(), show=False)
            plt.savefig(f"shap_f2_{model_type}.png")
            mlflow.log_artifact(f"shap_f2_{model_type}.png")
        except Exception: pass

        mlflow.sklearn.log_model(model, "model", registered_model_name=f"Churn_F2_{model_type}")

for m in ["xgboost", "random_forest", "logistic_regression"]:
    run_telco_experiment(df_train, df_test, m)